# Modify and save an existing scenario

Open an existing TOML project, retrieve its complete scenario from the planning service, add a site to the exported dictionary, and save the modified scenario locally as a new project. The exported configuration contains embedded sites, so the new TOML does not require `nodes.csv`. The modified scenario is not executed.

## 1. Connect to the workspace

In [ ]:
from cisei_planning_sdk import PlanningClient, PlanningProject

client = PlanningClient(
    "http://network-service:8080",
    workspace="..",
    timeout=900,
)

## 2. Open the existing Wi-SUN scenario

In [ ]:
source_project = PlanningProject("../projects/graph_planner_wisun")
scenario = client.open_project(source_project, planner="graph")

scenario.summary

## 3. Retrieve the in-memory configuration

The service returns the complete expanded scenario. Sites originally loaded from `nodes.csv` are now embedded in the dictionary.

In [ ]:
config = scenario.export_scenario()

len(config["sites"])

## 4. Add a Wi-SUN leaf

The new site has one device and one concrete interface. Its identifiers are unique within the scenario, and `can_relay` is false because this node cannot forward traffic.

In [ ]:
config["sites"].append(
    {
        "site_id": "new_leaf",
        "kind": "leaf",
        "lat": -25.424500,
        "lon": -49.326500,
        "devices": [
            {
                "device_id": "new_leaf:d0",
                "site_id": "new_leaf",
                "connected": False,
                "can_route": False,
                "mount_height_m": 7.0,
            }
        ],
        "interfaces": [
            {
                "interface_id": "new_leaf:d0:i0",
                "device_id": "new_leaf:d0",
                "site_id": "new_leaf",
                "tech": "wisun",
                "freq_mhz": 915.0,
                "tx_power_dbm": 20.0,
                "antenna_id": "wisun_node_omni",
                "can_relay": False,
            }
        ],
    }
)

len(config["sites"])

## 5. Save the modified scenario

Save into a new project so the original example remains unchanged.

In [ ]:
destination_project = PlanningProject("../projects/graph_planner_wisun_extended")
scenario_path = destination_project.save_scenario(config)

scenario_path

## 6. Inspect the generated TOML

In [ ]:
print(scenario_path.read_text(encoding="utf-8"))

The new project can later be opened with `client.open_project(destination_project, planner="graph")`. Opening it would define the modified scenario on the server and allow the normal planning sequence.